In [10]:
import joblib
import mlflow
import pandas as pd
import os
import sys
from interpret import show

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from sklearn.metrics import confusion_matrix
sys.path.append(os.path.dirname(os.getcwd()))
from models.interface import model_interface
from data.DataLoader import DataLoader

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Retail Project")

<Experiment: artifact_location='./mlruns/1', experiment_id='1', lifecycle_stage='active', name='Retail Project', tags={}>

In [11]:
def find_mlruns_parent():
        """Search upward from current directory for a folder containing 'mlruns'."""
        current = os.path.abspath(".")
        while True:
                if os.path.exists(os.path.join(current, "mlruns")):
                        return current
                parent = os.path.dirname(current)
                if parent == current:
                        break
                current = parent
        raise FileNotFoundError("Could not find 'mlruns' directory. Please specify the path manually.")

mlruns_parent = find_mlruns_parent()

In [12]:
model_names = [
            "Logistic Regression",
            # "Support Vector Classifier",
            "XGBoost Classifier",
]

latest_runs = {}

for model_name in model_names:
        runs = mlflow.search_runs(
                experiment_names=["Retail Project"],
                filter_string=f"tags.task = 'classification' AND tags.model_name = '{model_name}' AND attribute.status = 'FINISHED'",
                order_by=["attribute.start_time DESC"],
                max_results=1,
        )

        if runs.empty:
                print(f"No run found for {model_name}")
                continue
        latest_runs[model_name] = runs.iloc[0]
        print(latest_runs[model_name])

run_id                                            b52a1fbfbec44c30ac7e9bfa0aa4e4f0
experiment_id                                                                    1
status                                                                    FINISHED
artifact_uri                     ./mlruns/1/b52a1fbfbec44c30ac7e9bfa0aa4e4f0/ar...
start_time                                        2026-09-02 15:13:35.398000+00:00
end_time                                          2026-09-02 15:13:36.564000+00:00
metrics.train_precision                                                   0.590226
metrics.train_f1                                                          0.518902
metrics.train_accuracy                                                    0.635215
metrics.test_precision                                                     0.45549
metrics.test_f1                                                           0.515539
metrics.test_recall                                                        0.63099
metr

In [13]:
models = {}
metrics_summary = []
for model_name, run in latest_runs.items():
    run_id = run.run_id
    
    # 1. Load the model
    model_file = os.path.join(mlruns_parent, "mlruns", "1", run_id, "artifacts", "model", "model.joblib")
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file not found: {model_file}")
    model = joblib.load(model_file)
    
    # 2. Get run metadata from MLflow
    mlflow_run = mlflow.get_run(run_id)
    
    # 3. Extract metrics (e.g., rmse, mae, r2)
    metrics = mlflow_run.data.metrics
    
    # 4. Extract hyperparameters
    params = mlflow_run.data.params
    
    # 5. Store everything in a structured dict
    models[model_name] = {
        'model': model,
        'metrics': metrics,
        'params': params
    }
    
    # 6. Collect for a summary DataFrame (optional)
    metrics_summary.append({
        'model': model_name,
        **metrics,
        **params
    })

# Create a summary DataFrame for easy comparison
summary_df = pd.DataFrame(metrics_summary)
summary_df.to_csv("result_data/regression_model_info.csv", index=False)

In [14]:
loader = DataLoader(
        excel_location="data/data.xlsx",
        classification_excel_location = "data/classification_data.xlsx",
        train_test_split_percentage=0.8
)

loader.load()
valid, log = loader.validate()
print(log)
if not valid:
        raise RuntimeError("Dataset validation failed.")

classification_train_df = loader.get_classificaation_train_dataframe()
classification_test_df = loader.get_classification_test_dataframe()

classification_test_df

--------------------- DataLoader Validation ---------------------
FIXES:
  [FIXED] Sheet 'FactOrder': column 'Geography Key' converted from 'int64' to 'str'.
  [FIXED] Sheet 'DimGeography': column 'Key' converted from 'int64' to 'str'.

Validation: PASSED


,Order ID,Segment,Order Priority,Order Date,Region,Market,Sales_sum,Sales_mean,Sales_std,Sales_median,...,Total_Profit,Discount_Amount,Sales_Per_Item,Profit_Per_Item,Category=Office Supplies,Category=Technology,Category=Furniture,Category_entropy,Category_mode,Ship Mode
0,CA-2012-141040,Consumer,High,"Tuesday, October 9, 2012",East,US,655.880022,327.940011,429.949223,327.940011,...,314.105003,0.000000,81.985003,39.263125,1,1,0,1.000000,Office Supplies,Second Class
1,IN-2012-39672,Consumer,High,"Tuesday, October 9, 2012",Oceania,APAC,176.201996,176.201996,0.000000,176.201996,...,56.742001,17.620200,88.100998,28.371000,0,0,1,0.000000,Furniture,Same Day
2,IN-2012-12141,Consumer,Medium,"Tuesday, October 9, 2012",North Asia,APAC,493.230000,164.410000,216.923312,47.279999,...,12.330000,0.000000,54.803333,1.370000,3,0,0,0.000000,Office Supplies,Standard Class
3,IN-2012-56017,Corporate,Medium,"Tuesday, October 9, 2012",North Asia,APAC,27.000000,27.000000,0.000000,27.000000,...,13.200000,0.000000,13.500000,6.600000,1,0,0,0.000000,Office Supplies,Second Class
4,ID-2012-57809,Corporate,Medium,"Tuesday, October 9, 2012",Southeast Asia,APAC,861.885788,430.942894,502.532436,430.942894,...,-559.664178,389.966560,172.377158,-111.932836,1,0,1,1.000000,Furniture,Standard Class
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5003,MX-2011-140648,Home Office,Medium,"Wednesday, September 7, 2011",North,LATAM,51.599998,51.599998,0.000000,51.599998,...,3.600000,0.000000,8.600000,0.600000,1,0,0,0.000000,Office Supplies,Standard Class
5004,RO-2011-2770,Home Office,Medium,"Wednesday, September 7, 2011",EMEA,EMEA,929.580025,232.395006,320.725777,83.054998,...,226.530006,0.000000,116.197503,28.316251,1,2,1,0.946395,Technology,Standard Class
5005,IZ-2011-890,Corporate,Medium,"Wednesday, September 7, 2011",EMEA,EMEA,135.959995,67.979998,45.565960,67.979998,...,18.120000,0.000000,22.659999,3.020000,2,0,0,0.000000,Office Supplies,Standard Class
5006,CA-2011-117765,Home Office,Medium,"Wednesday, September 7, 2011",Central,US,643.780003,160.945001,190.485467,97.010004,...,178.271002,0.000000,58.525455,16.206455,2,0,2,1.000000,Furniture,Standard Class


In [15]:
def confusion_matrix_chart(y_true, y_pred, class_names):
        cm = confusion_matrix(y_true, y_pred)
        
        print(cm)

        # Create figure
        fig = go.Figure(data=go.Heatmap(
                z=cm,
                x=class_names,
                y=class_names,
                text=cm,
                texttemplate="%{text}",
                textfont={"size": 14},
                colorscale="Blues",
                showscale=True,
                colorbar=dict(title="Count")
        ))

        # Add titles
        fig.update_layout(
                title=dict(
                        text="Confusion Matrix - Ship Mode Prediction",
                        font=dict(size=20),
                        x=0.5  # Center the title
                ),
                xaxis=dict(
                        title="Predicted Label",
                        title_font=dict(size=16)
                ),
                yaxis=dict(
                        title="True Label",
                        title_font=dict(size=16),
                        autorange="reversed"  # Standard convention
                ),
                width=700,
                height=600,
                template="plotly_white"
        )

        fig.show()

In [16]:
class_names = [
        "Standard Class",
        "Same Day",
        "Second Class",
        "First Class"
]

y_true, y_pred = models[model_names[0]]['model'].predict(classification_test_df)
confusion_matrix_chart(y_true, y_pred, class_names)

[[ 189    0    1  598]
 [  74    0    0  175]
 [ 152    0    0  848]
 [   0    0    0 2971]]


In [25]:
y_true, y_pred = models[model_names[1]]['model'].predict(classification_test_df)
confusion_matrix_chart(y_true, y_pred, class_names)

[[ 176  382    1  229]
 [  59  121    0   69]
 [ 131  416    2  451]
 [ 109  554    2 2306]]


In [32]:
xgm = models[model_names[1]]['model'].model
print(xgm)
importance = xgm.feature_importances_

# feature_names = X.columns
# for name, imp in sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)[:10]:
#     print(f"{name}: {imp:.4f}")
importance

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)


array([0.02387497, 0.02051098, 0.02174708, 0.02310334, 0.02114163,
       0.0233152 , 0.02076419, 0.0153982 , 0.02147628, 0.0207651 ,
       0.02407108, 0.02085275, 0.02231922, 0.02199768, 0.02448892,
       0.02507262, 0.02015346, 0.03046271, 0.01968516, 0.0201763 ,
       0.01383278, 0.01949988, 0.01705842, 0.02408494, 0.02439218,
       0.01559852, 0.01485022, 0.0124714 , 0.01713529, 0.01722372,
       0.        , 0.        , 0.01637926, 0.02003624, 0.02555984,
       0.01739206, 0.02283377, 0.03018986, 0.01825556, 0.23182924],
      dtype=float32)

In [18]:
# elestic_result = models[model_names[1]]['model'].predict(regression_test_df)
# pd.DataFrame({
#         "y_true": elestic_result[0],
#         "y_predict": elestic_result[1]
# }).to_csv("result_data/elestic_regression_result.csv", index=False)

# xgb_result = models[model_names[2]]['model'].predict(regression_test_df)
# pd.DataFrame({
#         "y_true": xgb_result[0],
#         "y_predict": xgb_result[1]
# }).to_csv("result_data/xgb_regression_result.csv", index=False)